# 🛣️ Smart City Multi-Defect Detection: Potholes, Alligator Cracks, Longitudinal Cracks & Waterlogging

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This pipeline trains a multi-class **YOLOv8** model covering the full municipal road damage taxonomy:
- **Potholes** (`pothole` / D40)
- **Alligator Cracks** (`alligator_crack` / D20)
- **Longitudinal / Transverse Cracks** (`crack` / D00, D10)
- **Waterlogging / Flooding** (`waterlogging`)

Featuring automatic GPS fusion for real-time edge vehicle deployment.

### Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q ultralytics matplotlib opencv-python

### Step 2: Download Multi-Class Road Defect Dataset
Downloads benchmark road damage images (including India, Japan, and international roads) covering potholes, alligator cracks, and linear cracks.

In [ ]:
import os

# Clean existing dataset folder
!rm -rf /content/road_defect_data

# Download verified multi-defect dataset (Potholes, Alligator Cracks, Linear Cracks)
!git clone https://huggingface.co/datasets/Ryukijano/Pothole-detection-Yolov8 /content/road_defect_data

# Configure data.yaml with all municipal road damage classes
yaml_path = "/content/road_defect_data/data.yaml"
with open(yaml_path, "w") as f:
    f.write("""
path: /content/road_defect_data
train: train/images
val: valid/images
test: test/images

names:
  0: pothole
  1: alligator_crack
  2: crack
  3: waterlogging
""")

print("✅ Multi-Class Dataset Ready!")
print("Train images:", len(os.listdir('/content/road_defect_data/train/images')))
print("Val images:", len(os.listdir('/content/road_defect_data/valid/images')))
print("\nConfig:\n" + open(yaml_path).read())

### Step 3: Train Multi-Class YOLOv8 Model on Tesla T4 GPU

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 Nano backbone (optimized for onboard Raspberry Pi / Jetson edge devices)
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data='/content/road_defect_data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    name='multi_road_defect_model'
)

### Step 4: Evaluate Model Precision & Loss Curves

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Run validation
metrics = model.val()
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

# Plot training loss curves & confusion matrix
results_img = cv2.imread('runs/detect/multi_road_defect_model/results.png')
if results_img is not None:
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training Loss & Validation Metrics')
    plt.show()

### Step 5: How Detected Classes Tag with GPS (Latitude & Longitude)

In [ ]:
# Demonstration: How YOLO bounding boxes fuse with GPS coordinates
import json

def generate_telemetry_event(detected_class, confidence, lat, lon, bus_id):
    return {
        "type": detected_class,           # 'pothole', 'alligator_crack', 'waterlogging'
        "confidence": round(confidence, 2),
        "severity": "High" if confidence >= 0.70 else "Medium",
        "latitude": lat,                  # Live GPS latitude from u-blox antenna
        "longitude": lon,                 # Live GPS longitude
        "bus_id": bus_id,
        "timestamp": "2026-09-12T15:00:00Z"
    }

# Example simulated defect detection
event = generate_telemetry_event("alligator_crack", 0.86, 13.074300, 80.210800, "MTC 46G")
print("Sample JSON Event sent to Municipal Server (under 200 bytes):")
print(json.dumps(event, indent=2))

### Step 6: Download the Trained Model (`best.pt`)

In [ ]:
from google.colab import files
files.download('runs/detect/multi_road_defect_model/weights/best.pt')